# 🪐 Planet or Impostor?
## Vetting TESS Transit Signals with Python and LEO-Vetter

---
## What You'll Learn

In this workshop, you will:
1. Download a **real TESS light curve** and watch a transit signal appear
2. Learn why **most dips in starlight are *not* planets**
3. Write your own **vetting tests** to tell planets from eclipsing binaries, variable stars and instrumental glitches
4. Use **LEO-Vetter**, a professional automated vetting tool, and compare its verdicts to yours
5. (Take-home) Vet the **real star** from Part 1 yourself

**You should be comfortable reading short Python snippets and filling in a few lines.**

### Workshop roadmap

| Time | Section | What you do |
|------|---------|-------------|
| 0-5 min | Background and setup | Read, run two cells |
| 5-20 min | **Part 1: A real TESS light curve** | *Run only.* Real data becomes a planet candidate |
| 20-45 min | **Part 2a: Vetting by hand** | 4 short exercises on 5 simulated stars (4 impostors, 1 planet) |
| 45-60 min | **Part 2b: LEO-Vetter** | Run the pro tool, compare with your verdicts, discuss when a signal *could* be a planet |
| Take-home | **Part 3: Vet the real star** | Apply everything to the TESS data from Part 1 |

---
## Background: Why Vetting Matters

### The transit method

When a planet passes in front of its star, it blocks a tiny fraction of the starlight. The star dims a little, stays dim for a few hours, then brightens again. If we watch for long enough, the dip **repeats every orbit**.

The depth of the dip tells us how big the planet is compared to its star:

$$\text{depth} \approx \left(\frac{R_{\text{planet}}}{R_{\text{star}}}\right)^2$$

TESS (the Transiting Exoplanet Survey Satellite) watches tens of thousands of stars at a time, so software finds **thousands** of repeating dips. But a repeating dip is only a *candidate*.

### The usual suspects

Many things besides planets make a star dim periodically. **Vetting** means testing each candidate against these impostors:

| Impostor | What is really happening | Telltale sign | Test we'll use |
|----------|-------------------------|---------------|----------------|
| **Eclipsing binary** | Two stars orbit each other and eclipse | Alternating odd/even depths; a dip *between* transits; too big to be a planet | Size, odd/even, secondary eclipse |
| **Stellar variability** | Starspots rotating in and out of view | A smooth wave, not a sharp box | Compare with a sine-wave periodogram |
| **Instrumental systematic** | Scattered light or spacecraft events | Dips line up with the background flux | Background check |
| **Nearby eclipsing star** | A faint binary blended into the same pixels | The dimming comes from a different position on the sky | Pixel-level (difference imaging), discussed only |

**No single test settles the question.** Professionals combine many.

---
## Setup: Loading Libraries

- `numpy`: math with arrays (lists of numbers)
- `matplotlib`: plots
- `lightkurve`: the standard Python library for TESS and Kepler light curves
- `leo-vetter`: automated vetting of transit signals (we meet it in Part 2b)

In [ ]:
# Install the libraries (takes about a minute the first time)
!pip install -q leo-vetter lightkurve

import numpy as np
import matplotlib.pyplot as plt
import lightkurve as lk
import warnings
warnings.filterwarnings("ignore")

meaning_of_life = 42

np.random.seed(meaning_of_life)  # so everyone gets the same random numbers

print("✓ Setup complete!")

### Where does the data come from?

By default, `lightkurve` downloads TESS data from **MAST**, NASA's online archive. If MAST is slow or down, we have a backup copy of the data in the workshop's GitHub repo.

**Only change these settings if the download in Part 1 fails** (your instructor will tell you):

| Setting | When to use it |
|---------|----------------|
| `USE_BACKUP = True` | MAST is failing. Fetches a small backup file from the workshop repo. No sign-in needed. |
| `USE_DRIVE = True` | The backup fetch also fails (for example, GitHub is blocked on your network). You'll download the file by hand, put it in your Google Drive, and authorize access. |

In [ ]:
# ======================= DATA SOURCE SETTINGS =======================
USE_BACKUP = False   # <- set to True if the MAST download fails
USE_DRIVE  = False   # <- set to True only if USE_BACKUP also fails

# Instructor: fill these in once the repo exists
REPO_RAW     = "https://raw.githubusercontent.com/UBCAstronomyClub/Exoplanet_Vetting/main/data"
DRIVE_FOLDER = "/content/drive/MyDrive/Colab Notebooks/workshop4"   # where you put the downloaded .npz file (USE_DRIVE only)

---
# Part 1: A Real TESS Light Curve

**This part is run-only.** Press shift+enter through the cells and read the note under each plot. The goal is to *understand* what each plot shows. The code stays here for you to read, tweak and reuse.

Our star is **TIC 287196418**, observed by TESS in **Sector 14** (about 27 days in 2019). A *TIC ID* is the star's number in the TESS Input Catalog.

### Step 1: Download the light curve

A light curve is a star's brightness over time. The next cell asks MAST for this star's data from the SPOC pipeline (the official TESS data processing pipeline), then:
- removes missing points (`remove_nans`)
- divides by the median so that 1.0 means "the star's typical brightness" (`normalize`)

> 💡 If this cell fails, scroll up, set `USE_BACKUP = True`, and re-run **both** cells.

In [ ]:
import urllib.request

TIC, SECTOR = 287196418, 14

def load_from_npz(path):
    # Rebuild a light curve from a saved backup file
    d = np.load(path)
    lc = lk.LightCurve(time=d["time"], flux=d["flux"], flux_err=d["flux_err"])
    lc["sap_bkg"] = d["bkg"]
    return lc

def get_lightcurve(tic, sector):
    name = f"tic{tic}_s{sector}.npz"
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        return load_from_npz(f"{DRIVE_FOLDER}/{name}")
    if USE_BACKUP:
        urllib.request.urlretrieve(f"{REPO_RAW}/{name}", name)
        return load_from_npz(name)
    try:
        lc = lk.search_lightcurve(f"TIC {tic}", author="SPOC", sector=sector, exptime=120).download()
    except Exception as err:
        raise RuntimeError("Could not download from MAST. Set USE_BACKUP = True in the settings cell "
                           "and re-run both cells.") from err
    if lc is None:
        raise RuntimeError("MAST returned no data. Set USE_BACKUP = True in the settings cell "
                           "and re-run both cells.")
    return lc.remove_nans().normalize()

lc_part1 = get_lightcurve(TIC, SECTOR)
print(f"✓ Loaded {len(lc_part1)} measurements of TIC {TIC} (Sector {SECTOR})")
print(f"Time between measurements: {np.median(np.diff(lc_part1.time.value)) * 24 * 60:.1f} minutes")

In [ ]:
ax = lc_part1.scatter(s=2, color="k", alpha=0.4)
ax.set_title(f"TIC {TIC}: raw light curve")
plt.show()

**What you're looking at:** The star's brightness wobbles up and down roughly once a day, by about ±0.4%. That is most likely starspots rotating in and out of view. You can already see sharp dips that don't belong to the wobble, about every 3.7 days. Our job is to separate those dips from the wobble, so we remove the wobble first.

### Step 2: Bin and flatten

TESS records a point every 2 minutes, which is far more detail than we need. **Binning** averages neighboring points into 10-minute bins, which makes the data lighter and less noisy.

**Flattening** then fits a smooth curve to the slow wobble and divides it out. The `window_length` counts **data points, not days**. After binning, 181 points × 10 minutes ≈ 30 hours: much longer than a transit (a few hours), but shorter than the starspot wobble.

In [ ]:
lc_bin = lc_part1.bin(time_bin_size=10 / 60 / 24).remove_nans()   # 10-minute bins
flat = lc_bin.flatten(window_length=181, polyorder=2, break_tolerance=50)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].scatter(lc_bin.time.value, lc_bin.flux.value, s=3, c="k", alpha=0.5)
axes[0].set_ylabel("Binned flux")
axes[1].scatter(flat.time.value, flat.flux.value, s=3, c="k", alpha=0.5)
axes[1].set_ylabel("Flattened flux")
axes[1].set_xlabel("Time (days)")
axes[0].set_title("Before and after flattening")
plt.tight_layout()
plt.show()

**What you're looking at:** Flattening reduces the wobble to about half its size, so the dips stand out much more clearly. Some wobble is left, because the filter can't follow a one-day cycle without also erasing the transits (they only last a few hours). That is a real trade-off in transit searches: a window long enough to protect the transits also leaves some variability behind. The first few points at the far left are an edge effect of the filter, not a real dip.

### Step 3: Search for the period (Box Least Squares)

**BLS** slides a box-shaped dip across the data at thousands of trial periods and asks: "how well does a repeating box at this period explain the data?" The period with the most power is the best guess. We search periods between 1 and 5 days, as in the earlier exoplanet workshop.

In [ ]:
PERIOD_GRID_REAL = np.linspace(1, 5, 10000)        # periods the search will try (days)
DURATION_GRID = [0.05, 0.08, 0.11, 0.15, 0.2]      # transit durations the search will try (days)

bls = flat.to_periodogram(method="bls", period=PERIOD_GRID_REAL, duration=DURATION_GRID)

ax = bls.plot()
ax.set_title(f"BLS periodogram for TIC {TIC}")
plt.show()

print(f"Best period:  {bls.period_at_max_power.value:.4f} days")
print(f"Mid-transit:  {float(bls.transit_time_at_max_power.value):.3f} (a reference transit time)")
print(f"Duration:     {bls.duration_at_max_power.value:.2f} days")

**What you're looking at:** One tall peak at the orbital period. Smaller peaks at simple fractions or multiples of it are *aliases*. We'll see later that aliases are one way eclipsing binaries sneak in.

### Step 4: Fold the light curve

Cut the light curve into pieces one period long and stack them. All the transits line up. This is **phase folding**.

In [ ]:
period = float(bls.period_at_max_power.value)
epoch = float(bls.transit_time_at_max_power.value)

folded = flat.fold(period=period, epoch_time=epoch)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(folded.time.value, folded.flux.value, s=3, c="gray", alpha=0.4, label="every observation")
# average the points in 0.02-day (about 30-minute) slices of phase
x, y = folded.time.value, folded.flux.value
edges = np.arange(-0.6, 0.6001, 0.02)
centers = 0.5 * (edges[:-1] + edges[1:])
means = [np.nanmean(y[(x >= lo) & (x < hi)]) for lo, hi in zip(edges[:-1], edges[1:])]
ax.plot(centers, means, "r.-", label="binned average")
ax.set_xlim(-0.6, 0.6)
ax.set_xlabel("Days from mid-transit")
ax.set_ylabel("Flattened flux")
ax.set_title(f"TIC {TIC}, folded at the BLS period")
ax.legend()
plt.show()

**What you're looking at:** Hundreds of noisy points collapse into one clean dip. That is what a transit looks like.

### ✅ We have a candidate. Now what?

We now know a **period**, a **mid-transit time** and a **duration**. That is everything an automated search produces. But nothing so far tells us this is a planet, and pipelines find thousands of signals like this. **Most are impostors.**

You'll vet this star yourself in the take-home (Part 3). First we need practice on cases where we *know* the answer, so for Part 2 we switch to **simulated** stars.

### The same four steps, bundled into one function

This function repeats exactly what you just ran (clean, bin, flatten, BLS) and packages the result in a Python dictionary. We'll reuse it for every star from here on.

In [ ]:
def arr(x):
    # Turn lightkurve/astropy columns into plain NumPy arrays (LEO-Vetter needs plain arrays)
    x = getattr(x, "value", x)
    if hasattr(x, "filled"):
        x = x.filled(np.nan)
    return np.asarray(x, dtype=float)

PERIOD_GRID = np.linspace(1, 8, 12000)   # a wider search, used for the simulated stars

def process_lightcurve(lc, name="star", period_grid=PERIOD_GRID, duration_grid=DURATION_GRID):
    # Clean -> bin to 10 min (if finer) -> flatten -> BLS. Returns everything the vetting tests need.
    lc = lc.remove_nans().normalize()
    if np.median(np.diff(lc.time.value)) * 24 * 60 < 9:       # finer than ~10-minute cadence
        lc = lc.bin(time_bin_size=10 / 60 / 24).remove_nans()
    flat = lc.flatten(window_length=181, polyorder=2, break_tolerance=50)
    bls = flat.to_periodogram(method="bls", period=period_grid, duration=duration_grid)
    bkg = arr(lc["sap_bkg"]) if "sap_bkg" in lc.colnames else np.zeros(len(lc))
    bkg = np.where(np.isfinite(bkg), bkg, np.nanmedian(bkg) if np.isfinite(bkg).any() else 0.0)
    return {"name": name, "time": arr(flat.time), "raw": arr(lc.flux), "flux": arr(flat.flux),
            "err": arr(flat.flux_err), "bkg": bkg,
            "per": float(bls.period_at_max_power.value),
            "epo": float(bls.transit_time_at_max_power.value),
            "dur": float(bls.duration_at_max_power.value)}

print("✓ process_lightcurve() ready")

---
# Part 2a: Vetting by Hand

## The Data: Five Simulated Stars

For practice we use **simulated** light curves that look like TESS data. Why simulate?

1. We **know the true answer** for every star (like the true CMB in the previous workshop)
2. Nothing to download, so nothing can break
3. They have the same format as real TESS light curves, so everything you write works on real data too

Each star is observed for about 27 days with a gap in the middle, plus starspot wobble and noise. We'll pull out candidate signals with the same `process_lightcurve()` you just met.

> 🙈 **Do not read the next cell if you want the challenge to stay fun.** It contains the simulation code, and it gives away what each star really is.

In [ ]:
# ======================= SIMULATION HELPERS (you do not need to read this) =======================
import batman

T_START = 1684.0  # days (TESS Julian date)

def make_time(days=27.4, cadence_min=10.0, gap=(13.2, 14.3)):
    t = T_START + np.arange(0, days, cadence_min / 1440.0)
    keep = ~((t - T_START > gap[0]) & (t - T_START < gap[1]))
    return t[keep]

def planet_signal(t, per, t0, rp, a, inc):
    p = batman.TransitParams()
    p.t0, p.per, p.rp, p.a, p.inc = t0, per, rp, a, inc
    p.ecc, p.w, p.u, p.limb_dark = 0.0, 90.0, [0.4, 0.25], "quadratic"
    return batman.TransitModel(p, t).light_curve(p) - 1.0

def eclipse_signal(t, per, t0, depth, dur, phase=0.0, ingress_frac=0.25):
    # Trapezoid eclipse. A larger ingress_frac makes a more V-shaped (grazing) eclipse.
    ph = np.mod(t - t0 - phase * per + 0.5 * per, per) - 0.5 * per
    half, ing = 0.5 * dur, ingress_frac * dur
    x = np.abs(ph)
    out = np.zeros_like(t)
    out[x <= (half - ing)] = -depth
    slope = (x > (half - ing)) & (x < half)
    out[slope] = -depth * (half - x[slope]) / ing
    return out

def starspot_wobble(t, rng, prot, amp):
    ph = rng.uniform(0, 2 * np.pi, 2)
    return amp * (np.sin(2 * np.pi * t / prot + ph[0]) + 0.35 * np.sin(4 * np.pi * t / prot + ph[1]))

def scattered_light(t, rng, spike_times=None, spike_amp=0.0):
    # Background flux: a slow ramp every 13.7-day orbit, plus wiggles and (optionally) spikes
    ph = np.mod(t - T_START, 13.7) / 13.7
    bkg = 1400 + 5500 * np.sin(np.pi * ph) ** 2 + 200 * np.sin(2 * np.pi * (t - T_START) / 1.1)
    bkg += rng.normal(0, 25, len(t))
    if spike_times is not None:
        for ts in spike_times:
            bkg += spike_amp * np.exp(-0.5 * ((t - ts) / 0.12) ** 2)
    return bkg

STAR_SPECS = {
    "A":  dict(kind="planet",             seed=11, per=3.70, t0=1685.1, rp=0.060, a=11.0, inc=88.6),
    "B":  dict(kind="binary_alternating", seed=12, per=3.45, t0=1684.9, depth1=0.010, depth2=0.006, dur=0.13),
    "C":  dict(kind="binary_secondary",   seed=13, per=5.10, t0=1686.0, depth1=0.013, depth2=0.003, dur=0.16),
    "D":  dict(kind="variable",           seed=14, prot=2.9, amp=0.012),
    "E":  dict(kind="systematic",         seed=15, per=4.60, t0=1685.6, dip=0.003),
    "M1": dict(kind="binary_alternating", seed=21, per=2.60, t0=1685.2, depth1=0.009, depth2=0.0055, dur=0.12),
    "M2": dict(kind="planet",             seed=22, per=2.35, t0=1684.7, rp=0.050, a=8.5, inc=88.3),
}

def simulate_star(spec):
    rng = np.random.default_rng(spec["seed"])
    t = make_time()
    kind = spec["kind"]
    signal = np.zeros_like(t)
    spikes, spike_amp = None, 0.0
    prot = spec.get("prot", rng.uniform(4.0, 6.0))
    amp = spec.get("amp", 0.0025)
    if kind == "planet":
        signal = planet_signal(t, spec["per"], spec["t0"], spec["rp"], spec["a"], spec["inc"])
    elif kind == "binary_alternating":
        signal = (eclipse_signal(t, spec["per"], spec["t0"], spec["depth1"], spec["dur"])
                  + eclipse_signal(t, spec["per"], spec["t0"], spec["depth2"], spec["dur"], phase=0.5))
    elif kind == "binary_secondary":
        signal = (eclipse_signal(t, spec["per"], spec["t0"], spec["depth1"], spec["dur"], ingress_frac=0.40)
                  + eclipse_signal(t, spec["per"], spec["t0"], spec["depth2"], spec["dur"], phase=0.5, ingress_frac=0.40))
    elif kind == "systematic":
        spikes = spec["t0"] + spec["per"] * np.arange(0, 6)
        spikes = spikes[spikes < t.max()]
        spike_amp = 2500.0
        for ts in spikes:
            signal += -spec["dip"] * np.exp(-0.5 * ((t - ts) / 0.05) ** 4)
    wobble = starspot_wobble(t, rng, prot, amp)
    noise = 1000e-6
    flux = 1.0 + wobble + signal + rng.normal(0, noise, len(t))
    return {"time": t, "flux": flux, "err": np.full(len(t), noise),
            "bkg": scattered_light(t, rng, spikes, spike_amp)}

def prepare_star(name):
    # Simulate one star, then run it through the same process_lightcurve() used on real data
    d = simulate_star(STAR_SPECS[name])
    lc = lk.LightCurve(time=d["time"], flux=d["flux"], flux_err=d["err"])
    lc["sap_bkg"] = d["bkg"]
    return process_lightcurve(lc, name=name)

print("✓ Simulation helpers loaded!")

### Five candidates

Each of these stars passed through the same pipeline you ran in Part 1 (clean, bin, flatten, BLS). Here is what came out:

In [ ]:
stars = {name: prepare_star(name) for name in ["A", "B", "C", "D", "E"]}

print(f"{'Star':<6}{'Period (d)':<14}{'Duration (d)':<14}")
print("-" * 34)
for name, s in stars.items():
    print(f"{name:<6}{s['per']:<14.3f}{s['dur']:<14.2f}")

Four of these are impostors. One is a planet. Our job is to work out which. Every test needs to know how far each observation is from the nearest predicted mid-transit. This function gives that, in days (negative means before the transit, positive means after).

| Test | Question | Catches |
|------|----------|---------|
| 1. Size | Is the object a planetary size? | Large eclipsing binaries |
| 2. Odd/even | Are odd and even transits the same depth? | Eclipsing binaries |
| 3. Secondary | Is there a dip *between* the transits? | Eclipsing binaries |
| 4. Background | Does the dip line up with background light? | Instrumental systematics |

In [ ]:
def phase_days(time, per, epo):
    # Days from the nearest predicted mid-transit
    return (time - epo + 0.5 * per) % per - 0.5 * per

def plot_fold(star, ax=None, title=None):
    # Quick look at a star folded at its candidate period
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 3.5))
    ph = phase_days(star["time"], star["per"], star["epo"])
    ax.scatter(ph, star["flux"], s=2, c="gray", alpha=0.4)
    ax.set_xlim(-0.6, 0.6)
    ax.set_xlabel("Days from mid-transit")
    ax.set_ylabel("Flux")
    ax.set_title(title or f"Star {star['name']}  (P = {star['per']:.3f} d)")

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey=True)
for ax, name in zip(axes.flat, "ABCDE"):
    plot_fold(stars[name], ax)
axes.flat[5].axis("off")
plt.tight_layout()
plt.show()

Five candidates, five repeating dips. A, B and C show clear dips of different shapes, E shows a dip like A's, and D shows almost nothing. All five were reported by the same search. Which are planets? That's the vetting question.

---
## Exercise 1: The Size Test

**Idea:** A planet blocks light in proportion to its area, so the depth gives its radius. If the implied radius is bigger than any planet could be (more than about 2 Jupiter radii), the object is a star.

$$R_{\text{planet}} = R_{\text{star}} \times \sqrt{\text{depth}}$$

**What you need to do** (assume every simulated star is Sun-sized; 1 solar radius = 109.1 Earth radii):
1. Find which observations are **in transit** (within half a duration of mid-transit)
2. Find which are **out of transit** (more than one full duration away)
3. Depth = typical out-of-transit flux minus typical in-transit flux (use the **median**)
4. Convert depth to radius in Earth radii

In [ ]:
def planet_radius(star, r_star_rsun=1.0):
    # Estimate the radius of the transiting object, in Earth radii
    phase = phase_days(star["time"], star["per"], star["epo"])
    flux = star["flux"]

    # TODO 1: True for points within half a transit duration of mid-transit
    #         Hint: np.abs(phase) < 0.5 * star["dur"]
    in_transit = None

    # TODO 2: True for points more than one full duration away from mid-transit
    out_of_transit = None

    # TODO 3: depth = median(out-of-transit flux) - median(in-transit flux)
    depth = None

    # TODO 4: Rp/Rs = sqrt(depth), then multiply by the star's radius and by 109.1
    rp_earth = None

    return depth, rp_earth

<details>
<summary>💡 Hint 1: the idea</summary>

A transit is a time when the star is **dimmer than usual**, so to measure the dip you need two groups of points: ones *inside* the dip and ones *outside* it.

The variable `phase` tells you where each point sits: it is the number of days from the nearest mid-transit. `phase = 0` is the middle of a transit, and `phase = ±0.05` is a little before or after it.
</details>

<details>
<summary>💡 Hint 2: finding the two groups</summary>

You want a True/False array (a "mask") for each group. A comparison on a whole array gives one, for example `phase < 0.1`.

Points on **either side** of the mid-transit should count the same way, so compare the *distance* from zero: `np.abs(phase)`.

- In transit: the distance is smaller than **half** the transit duration, `star["dur"]`.
- Out of transit: the distance is larger than a **full** duration. (Skipping the zone in between keeps the edges of the dip out of your baseline.)
</details>

<details>
<summary>💡 Hint 3: depth and radius</summary>

- **Depth** = (typical flux outside the transit) − (typical flux inside it). Use `np.median(flux[mask])` for "typical flux". Put the out-of-transit term first so the depth comes out positive.
- A transit blocks (Rp/R★)² of the light, so Rp/R★ = `np.sqrt(depth)`.
- Multiply by the star's radius in solar radii (`r_star_rsun`), then by **109.1** to convert from solar radii to Earth radii.
</details>

<details>
<summary>💡 Hint 4: nearly there (fill in the blanks)</summary>

```python
in_transit = np.abs(phase) < ____ * star["dur"]
out_of_transit = np.abs(phase) > ____ * star["dur"]
depth = np.median(flux[____]) - np.median(flux[____])
rp_earth = np.sqrt(____) * r_star_rsun * ____
```

**Self-check:** you should get a few to about 10 Earth radii for Stars A, B and C. If you see `nan`, your depth is negative, so the subtraction is the wrong way round.
</details>

<details>
<summary>✅ Solution</summary>

```python
def planet_radius(star, r_star_rsun=1.0):
    # Estimate the radius of the transiting object, in Earth radii
    phase = phase_days(star["time"], star["per"], star["epo"])
    flux = star["flux"]

    # Points within half a transit duration of mid-transit
    in_transit = np.abs(phase) < 0.5 * star["dur"]

    # Points more than one full duration away from mid-transit
    out_of_transit = np.abs(phase) > star["dur"]

    # Depth = how much dimmer the star is during the transit
    depth = np.median(flux[out_of_transit]) - np.median(flux[in_transit])

    # Rp/Rs = sqrt(depth); 1 solar radius = 109.1 Earth radii
    rp_earth = np.sqrt(depth) * r_star_rsun * 109.1

    return depth, rp_earth
```
</details>

In [ ]:
# Test your size function
try:
    print(f"{'Star':<6}{'Depth (ppt)':<14}{'Radius (Earth radii)':<22}")
    print("-" * 42)
    for name in ["A", "B", "C"]:
        depth, rp = planet_radius(stars[name])
        print(f"{name:<6}{depth*1e3:<14.2f}{rp:<22.1f}")
    print("\nFor scale: Neptune = 3.9, Jupiter = 11.2 Earth radii. LEO-Vetter treats anything over 22 as 'too large'.")
    print("\n🤔 Does any star fail the size test? Note which ones pass, but don't trust them yet.")
except TypeError:
    print("❌ Please fill in all four None values in planet_radius() above!")

---
## Exercise 2: The Odd/Even Test

**Idea:** If two stars eclipse each other, the *first* star passing in front of the second usually gives a different dip than the *second* passing in front of the first. If we split the transits into odd-numbered and even-numbered ones, a real planet gives the same depth in both. A binary that BLS has mistaken for a half-period signal does not.

**What you need to do:**
1. Give every observation a **transit number** (0, 1, 2, ...) by rounding `(time - epo) / per`
2. Odd transits: in transit **and** odd transit number. Even transits: in transit **and** even transit number
3. Measure the depth of each group
4. Compute how many **sigma** apart they are. A difference above about 3 sigma is unlikely to be chance.

In [ ]:
def odd_even_test(star):
    time, flux = star["time"], star["flux"]
    phase = phase_days(time, star["per"], star["epo"])
    in_transit = np.abs(phase) < 0.5 * star["dur"]
    out_of_transit = np.abs(phase) > star["dur"]
    baseline = np.median(flux[out_of_transit])
    scatter = np.std(flux[out_of_transit])

    # TODO 1: Transit number of every point (round to the nearest whole number, as integers)
    #         Hint: np.round((time - star["epo"]) / star["per"]).astype(int)
    transit_number = None

    # TODO 2: Masks for odd and even transits
    #         Hint: transit_number % 2 == 1 is True for odd numbers
    odd = None
    even = None

    # TODO 3: Depth of each group = baseline - median flux of that group
    depth_odd = None
    depth_even = None

    # Uncertainty on the difference (provided for you)
    err = scatter * np.sqrt(1 / np.sum(odd) + 1 / np.sum(even))

    # TODO 4: Difference in sigma = |depth_odd - depth_even| / err
    n_sigma = None

    return depth_odd, depth_even, n_sigma

<details>
<summary>💡 Hint 1: the idea</summary>

A planet gives the **same** dip every orbit. A binary that BLS has caught at half its true period alternates between a deep eclipse and a shallow one.

So split the dips into two groups, the 1st, 3rd, 5th... and the 2nd, 4th, 6th..., and compare how deep each group is. Similar depths are consistent with a planet. Different depths mean two different eclipses.
</details>

<details>
<summary>💡 Hint 2: numbering the transits</summary>

`time - star["epo"]` is the number of days since the reference transit. Divide that by the period and you get how many orbits have passed: 0.0 at the reference transit, 1.0 one period later, 2.0 after that, and fractions in between (like 0.4 for a point between two transits).

Round to the nearest whole number, so that every point gets the number of its closest transit: `np.round(...)`, then `.astype(int)`.
</details>

<details>
<summary>💡 Hint 3: picking odd and even</summary>

The remainder operator `%` finds odd and even numbers: `transit_number % 2` is 1 for odd and 0 for even.

You only want points that are *in* transit **and** in the right group. Combine two True/False arrays with `&` (not `and`), and use brackets: `in_transit & (something)`.
</details>

<details>
<summary>💡 Hint 4: depth and sigma</summary>

- Depth of a group = `baseline - np.median(flux[group])`. This is the same idea as Exercise 1.
- The difference in sigma is the size of the difference between the two depths divided by `err`. Use `abs(...)` so the order doesn't matter.
</details>

<details>
<summary>💡 Hint 5: nearly there (fill in the blanks)</summary>

```python
transit_number = np.round((time - star["epo"]) / ____).astype(int)
odd = in_transit & (transit_number % 2 == ____)
even = in_transit & (transit_number % 2 == ____)
depth_odd = baseline - np.median(flux[____])
depth_even = baseline - np.median(flux[____])
n_sigma = abs(____ - ____) / err
```

**Self-check:** Star A should be well under 3σ, Star B should be huge (about 28σ), and Star C should be small (about 1σ).
</details>

<details>
<summary>✅ Solution</summary>

```python
def odd_even_test(star):
    time, flux = star["time"], star["flux"]
    phase = phase_days(time, star["per"], star["epo"])
    in_transit = np.abs(phase) < 0.5 * star["dur"]
    out_of_transit = np.abs(phase) > star["dur"]
    baseline = np.median(flux[out_of_transit])
    scatter = np.std(flux[out_of_transit])

    # Transit number of every point
    transit_number = np.round((time - star["epo"]) / star["per"]).astype(int)

    # Masks for odd and even transits
    odd = in_transit & (transit_number % 2 == 1)
    even = in_transit & (transit_number % 2 == 0)

    # Depth of each group
    depth_odd = baseline - np.median(flux[odd])
    depth_even = baseline - np.median(flux[even])

    # Uncertainty on the difference (provided for you)
    err = scatter * np.sqrt(1 / np.sum(odd) + 1 / np.sum(even))

    # Difference in sigma
    n_sigma = abs(depth_odd - depth_even) / err

    return depth_odd, depth_even, n_sigma
```
</details>

In [ ]:
# Test your odd/even function
try:
    print(f"{'Star':<6}{'Odd depth (ppt)':<18}{'Even depth (ppt)':<19}{'Difference (sigma)':<20}")
    print("-" * 63)
    for name in ["A", "B", "C"]:
        d_odd, d_even, nsig = odd_even_test(stars[name])
        flag = "  ⚠️ SUSPICIOUS" if nsig > 3 else ""
        print(f"{name:<6}{d_odd*1e3:<18.2f}{d_even*1e3:<19.2f}{nsig:<20.1f}{flag}")
except TypeError:
    print("❌ Please fill in all the None values in odd_even_test() above!")

**Think about it:** Star B's BLS period is about **half** its real orbital period. At that wrong period, the primary and secondary eclipses get stacked into the same fold, and their different depths show up as odd vs even. Whenever BLS locks onto an alias, this test is your safety net.

---
## Exercise 3: The Secondary Eclipse Test

**Idea:** Two stars eclipse each other *twice* per orbit: once when the first is in front, and once (half an orbit later) when the second is in front. A planet gives off almost no light of its own, so there is almost no second dip. If there is a clear dip at **phase 0.5** (halfway between transits), suspect a binary.

**What you need to do:**
1. Compute the **orbit fraction** of every point: `((time - epo) / per) % 1`. The transit sits at 0, and a secondary would sit at 0.5.
2. Mark points within half a duration of 0.5 as `secondary`
3. Measure the secondary depth against the baseline
4. Convert to sigma

In [ ]:
def secondary_test(star):
    time, flux = star["time"], star["flux"]
    per, dur = star["per"], star["dur"]

    # TODO 1: Orbit fraction: 0 at the transit, 0.5 halfway round the orbit
    #         Hint: ((time - star["epo"]) / per) % 1
    orbit_fraction = None

    # TODO 2: True for points within half a duration of orbit fraction 0.5
    #         Hint: np.abs(orbit_fraction - 0.5) < 0.5 * dur / per
    secondary = None

    # Baseline = points that are in neither the primary nor the secondary (provided for you)
    primary = np.abs(phase_days(time, per, star["epo"])) < dur
    baseline_mask = ~primary & ~secondary
    baseline = np.median(flux[baseline_mask])
    scatter = np.std(flux[baseline_mask])

    # TODO 3: Secondary depth = baseline - median flux during the secondary
    depth_sec = None

    # TODO 4: Significance in sigma = depth_sec / (scatter / sqrt(number of secondary points))
    n_sigma = None

    return depth_sec, n_sigma

<details>
<summary>💡 Hint 1: the idea</summary>

Two stars eclipse each other **twice** per orbit: once when the first is in front, and again half an orbit later when the second is in front. A planet gives off almost no light of its own, so there is almost no second dip.

So after the main dip (orbit fraction 0), look **halfway round the orbit** (orbit fraction 0.5) and ask: is there a dip there?
</details>

<details>
<summary>💡 Hint 2: orbit fraction</summary>

`(time - star["epo"]) / per` is the number of orbits since the reference transit: 0.0 at the transit, 1.0 one period later, and 0.5 halfway between.

Keep only the **fractional part** with `% 1`, so every point gets a number from 0 to 1. This is different from `phase_days`, which counts *days* from the nearest transit.
</details>

<details>
<summary>💡 Hint 3: picking the secondary points</summary>

You want points within half a transit duration of orbit fraction 0.5. Watch the units: `orbit_fraction` is a fraction of the *period*, but `dur` is in *days*. Convert the duration to the same units by dividing by the period: `dur / per`.

So the test is: the distance of `orbit_fraction` from 0.5, `np.abs(orbit_fraction - 0.5)`, is less than half of `dur / per`.
</details>

<details>
<summary>💡 Hint 4: depth and sigma</summary>

- Depth = `baseline - np.median(flux[secondary])`, as in the other exercises. A **positive** depth means a dip.
- Significance: the average of N points is uncertain by `scatter / sqrt(N)`, where N is the number of secondary points. `np.sum(secondary)` counts them. Then `n_sigma = depth_sec / (that uncertainty)`.
</details>

<details>
<summary>💡 Hint 5: nearly there (fill in the blanks)</summary>

```python
orbit_fraction = ((time - ____) / per) % ____
secondary = np.abs(orbit_fraction - ____) < 0.5 * ____ / per
depth_sec = baseline - np.median(flux[____])
n_sigma = depth_sec / (scatter / np.sqrt(np.sum(____)))
```

**Self-check:** Stars A and B should come out small (about 1σ and 2σ), and Star C should be large (about 18–19σ).
</details>

<details>
<summary>✅ Solution</summary>

```python
def secondary_test(star):
    time, flux = star["time"], star["flux"]
    per, dur = star["per"], star["dur"]

    # Orbit fraction: 0 at the transit, 0.5 halfway round the orbit
    orbit_fraction = ((time - star["epo"]) / per) % 1

    # Points within half a duration of orbit fraction 0.5
    secondary = np.abs(orbit_fraction - 0.5) < 0.5 * dur / per

    # Baseline = points that are in neither the primary nor the secondary (provided for you)
    primary = np.abs(phase_days(time, per, star["epo"])) < dur
    baseline_mask = ~primary & ~secondary
    baseline = np.median(flux[baseline_mask])
    scatter = np.std(flux[baseline_mask])

    # Secondary depth
    depth_sec = baseline - np.median(flux[secondary])

    # Significance in sigma
    n_sigma = depth_sec / (scatter / np.sqrt(np.sum(secondary)))

    return depth_sec, n_sigma
```
</details>

In [ ]:
# Test your secondary eclipse function
try:
    print(f"{'Star':<6}{'Secondary depth (ppt)':<24}{'Significance (sigma)':<22}")
    print("-" * 52)
    for name in ["A", "B", "C"]:
        d_sec, nsig = secondary_test(stars[name])
        flag = "  ⚠️ SUSPICIOUS" if nsig > 3 else ""
        print(f"{name:<6}{d_sec*1e3:<24.2f}{nsig:<22.1f}{flag}")
    print("\n🤔 Star B passes this test but failed the last one. Star C is the reverse.")
    print("   Neither test alone catches both binaries. That's why we use several.")
except TypeError:
    print("❌ Please fill in all the None values in secondary_test() above!")

---
## Exercise 4: The Background Test

**Idea:** TESS measures the star's brightness by adding up pixels, then subtracting the *background* (scattered light from Earth and the Moon, for instance). If the background changes suddenly, the star's measured brightness can glitch, creating dips that have nothing to do with the star.

So: **does the background jump up whenever the "transit" happens?**

**What you need to do:**
1. Define `nearby` points: between 1 and 3 durations from mid-transit (close in time, but outside the transit)
2. Compute the **excess** background during transit compared to nearby points
3. Convert it to a rough sigma. (This is a quick estimate, not a full statistical test.)

In [ ]:
def background_test(star):
    phase = phase_days(star["time"], star["per"], star["epo"])
    bkg = star["bkg"]
    in_transit = np.abs(phase) < 0.5 * star["dur"]

    # TODO 1: Points that are close in time but outside the transit
    #         Hint: (np.abs(phase) > star["dur"]) & (np.abs(phase) < 3 * star["dur"])
    nearby = None

    # TODO 2: Excess background in transit = mean(in transit) - mean(nearby)
    excess = None

    # Typical scatter of the nearby background (provided for you)
    local_scatter = np.std(bkg[nearby])

    # TODO 3: Rough significance = excess / (local_scatter / sqrt(number of in-transit points))
    n_sigma = None

    return excess, n_sigma

<details>
<summary>💡 Hint 1: the idea</summary>

TESS measures a star by adding up pixel brightness and then subtracting the **background** (scattered light from Earth and the Moon, for example). If the background jumps at the moment of a "transit", the star's measured brightness can glitch even though the star itself did nothing.

So ask: **is the background higher during the dip than it is just before and after?** We compare with *nearby* times and not the whole light curve, because the background drifts slowly over days.
</details>

<details>
<summary>💡 Hint 2: the nearby points</summary>

You want points that are close to the transit in time but not inside it. Use the distance from mid-transit, `np.abs(phase)`, and combine two conditions with `&` (and brackets):

- farther away than the transit's edge (more than **one** duration), and
- not too far away (less than **three** durations).

Leaving out the zone right next to the transit keeps the transit's own edges out of the comparison.
</details>

<details>
<summary>💡 Hint 3: the excess</summary>

Take the average background at the in-transit times and subtract the average background at the nearby times: `np.mean(bkg[mask])`. A **positive** answer means the background was higher during the dip.
</details>

<details>
<summary>💡 Hint 4: turning it into sigma</summary>

The average of N points is uncertain by about `scatter / sqrt(N)`. Here the scatter is `local_scatter`, and N is the number of in-transit points. `np.sum(in_transit)` counts how many `True` values the array has. Then:

n_sigma = excess / (the uncertainty on the average)
</details>

<details>
<summary>💡 Hint 5: nearly there (fill in the blanks)</summary>

```python
nearby = (np.abs(phase) > ____ * star["dur"]) & (np.abs(phase) < ____ * star["dur"])
excess = np.mean(bkg[____]) - np.mean(bkg[____])
n_sigma = excess / (local_scatter / np.sqrt(np.sum(____)))
```

**Self-check:** Stars A and C should come out near zero (about -0.2σ and 1.3σ), and Star E should be large (about 8-9σ). Anything above roughly 3σ is suspicious.
</details>

<details>
<summary>✅ Solution</summary>

```python
def background_test(star):
    phase = phase_days(star["time"], star["per"], star["epo"])
    bkg = star["bkg"]
    in_transit = np.abs(phase) < 0.5 * star["dur"]

    # Points that are close in time but outside the transit
    nearby = (np.abs(phase) > star["dur"]) & (np.abs(phase) < 3 * star["dur"])

    # Excess background in transit
    excess = np.mean(bkg[in_transit]) - np.mean(bkg[nearby])

    # Typical scatter of the nearby background (provided for you)
    local_scatter = np.std(bkg[nearby])

    # Rough significance
    n_sigma = excess / (local_scatter / np.sqrt(np.sum(in_transit)))

    return excess, n_sigma
```
</details>

In [ ]:
# Test your background function
try:
    print(f"{'Star':<6}{'Background excess':<22}{'Significance (sigma)':<22}")
    print("-" * 50)
    for name in ["A", "C", "E"]:
        excess, nsig = background_test(stars[name])
        flag = "  ⚠️ SUSPICIOUS" if abs(nsig) > 3 else ""
        print(f"{name:<6}{excess:<22.1f}{nsig:<22.1f}{flag}")
except TypeError:
    print("❌ Please fill in all the None values in background_test() above!")

### One more impostor: stellar variability (demo, run only; skip if short on time)

Star D has no transits at all, but BLS found a "signal" anyway. The giveaway: a tool made for **smooth waves** (a Lomb-Scargle periodogram), run on the *raw* light curve, finds a period simply related to BLS's.

In [ ]:
lc_D = lk.LightCurve(time=stars["D"]["time"], flux=stars["D"]["raw"], flux_err=stars["D"]["err"])
ls_D = lc_D.to_periodogram(method="lombscargle", minimum_period=1, maximum_period=8)

print(f"BLS period (the 'transit' signal): {stars['D']['per']:.2f} days")
print(f"Lomb-Scargle period (smooth wave):  {ls_D.period_at_max_power.value:.2f} days")
print(f"Ratio: {ls_D.period_at_max_power.value / stars['D']['per']:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
plot_fold(stars["D"], axes[0], title="Star D folded at the BLS period")
ph = (lc_D.time.value % ls_D.period_at_max_power.value) / ls_D.period_at_max_power.value
axes[1].scatter(ph, lc_D.flux.value, s=2, c="gray", alpha=0.4)
axes[1].set_title("Star D raw flux folded at the Lomb-Scargle period")
axes[1].set_xlabel("Orbit fraction")
plt.tight_layout()
plt.show()

**What you're looking at:** On the left, folding at BLS's period gives no clean, flat-bottomed dip, just leftover wobble. On the right, the raw light curve folded at the Lomb-Scargle period is one big, smooth wave, and its period is **twice** the one BLS reported. BLS was latching onto the wave's shape, not a transit. Real transits are sharp and flat-bottomed, so this is a starspot signal wearing a transit costume.

### 📝 Your verdicts

Before we bring in the professional tool, **commit to a verdict for each star** using what you've seen. Options: `"planet"`, `"eclipsing binary"`, `"variable star"`, `"systematic"`.

In [ ]:
# TODO: Fill in your verdicts, then run the cell
my_verdicts = {
    "A": None,
    "B": None,
    "C": None,
    "D": None,
    "E": None,
}
print(my_verdicts)

---
# Part 2b: Meet LEO-Vetter

You just wrote four vetting tests. Professional pipelines run **dozens**, automatically, on thousands of signals.

**LEO-Vetter** (the *Lazy Exoplanet Operations Vetter*) is a free, open-source Python tool for exactly this, written by Dr. Michelle Kunimoto and collaborators, and inspired by the Kepler mission's *Robovetter*. It works like this:

1. Compute a set of **metrics** from the light curve (signal strength, odd/even difference, secondary significance, planet radius, shape, and more)
2. Check each metric against a **pass/fail threshold**
3. Sort the signal into one of three bins:

| Verdict | Meaning |
|---------|---------|
| **PC** (planet candidate) | Passed every test |
| **FP** (astrophysical false positive) | A real eclipse signal, but not a planet (for example, an eclipsing binary) |
| **FA** (false alarm) | Not a real transit at all (for example, noise, variability, systematics) |

### Step 1: Describe the star

The tool needs to know about the *host star* (to turn depths into radii, and to check that the orbit makes physical sense). We'll give all our simulated stars Sun-like properties.

In [ ]:
from leo_vetter.main import TCELightCurve
from leo_vetter.thresholds import check_thresholds
from leo_vetter.stellar import quadratic_ldc
import io, contextlib

# Limb-darkening coefficients: how much the star's edge is dimmer than its center
u1, u2 = quadratic_ldc(5700.0, 4.4, 0.0)

sim_star_props = {
    "rad": 1.0,   "e_rad": 0.05,      # radius (solar radii) and uncertainty
    "mass": 1.0,  "e_mass": 0.05,     # mass (solar masses) and uncertainty
    "Teff": 5700.0, "e_Teff": 100.0,  # temperature (K) and uncertainty
    "rho": 1.41,                      # density (g/cm^3)
    "u1": float(u1), "u2": float(u2),
}

def make_tlc(star, tic=1):
    # Package one of our stars in the format LEO-Vetter expects
    return TCELightCurve(tic, star["time"], star["raw"], star["flux"], star["err"],
                         star["per"], star["epo"], star["dur"])

print("✓ LEO-Vetter loaded!")

### Step 2: Compute the metrics

`compute_flux_metrics` runs all the tests on a light curve (it fits transit models, compares odd and even transits, looks for a secondary, and more). Let's run it on **Star A**.

In [ ]:
tlc_A = make_tlc(stars["A"])
tlc_A.compute_flux_metrics(sim_star_props, verbose=False)

m = tlc_A.metrics
print(f"Signal-to-noise (MES):        {m['MES']:.1f}")
print(f"Number of transits:           {m['N_transit']:.0f}")
print(f"Planet radius:                {m['Rp']:.1f} Earth radii")
print(f"Odd/even difference:          {m['sig_dep']:.1f} sigma")
print(f"Secondary eclipse:            {m['sig_sec']:.1f} sigma")
print(f"Sinusoid strength (SWEET):    {m['sine_sig']:.1f}")

**Look familiar?** The radius, odd/even and secondary numbers are the same quantities you computed by hand. LEO-Vetter fits proper models instead of taking medians, and does many more checks.

### Step 3: Check the thresholds

`check_thresholds` compares all the metrics to cut-offs. With `verbose=True` it tells you **which tests failed**. We run it twice: once for false alarms ("FA") and once for astrophysical false positives ("FP").

In [ ]:
print("--- False alarm tests ---")
FA_A = check_thresholds(tlc_A.metrics, "FA", verbose=True)
print("\n--- Astrophysical false positive tests ---")
FP_A = check_thresholds(tlc_A.metrics, "FP", verbose=True)
print(f"\nFailed an FA test? {bool(FA_A)}    Failed an FP test? {bool(FP_A)}")

---
## Exercise 5: Turn the Test Results into a Verdict

`check_thresholds` returns `True` if the signal **failed at least one** test of that kind. Write the function that turns those two flags into a verdict.

**The rules:**
- Failed any false-alarm test → `"FA"`
- Otherwise, failed any astrophysical false-positive test → `"FP"`
- Otherwise → `"PC"`

In [ ]:
def leo_verdict(tlc):
    # Return "PC", "FP" or "FA" for a light curve whose metrics have been computed
    failed_FA = bool(check_thresholds(tlc.metrics, "FA"))
    failed_FP = bool(check_thresholds(tlc.metrics, "FP"))

    # TODO: Apply the rules above
    verdict = None

    return verdict

<details>
<summary>💡 Click here for the solution</summary>

```python

def leo_verdict(tlc):
    # Return "PC", "FP" or "FA" for a light curve whose metrics have been computed
    failed_FA = bool(check_thresholds(tlc.metrics, "FA"))
    failed_FP = bool(check_thresholds(tlc.metrics, "FP"))

    if failed_FA:
        verdict = "FA"
    elif failed_FP:
        verdict = "FP"
    else:
        verdict = "PC"

    return verdict

```
</details>

In [ ]:
# Test: Star A should have come out as a planet candidate
v = leo_verdict(tlc_A)
if v is None:
    print("❌ Please complete leo_verdict() above!")
elif v == "PC":
    print("✓ Star A is a planet candidate (PC). The function works!")
else:
    print(f"Star A got '{v}'. Check the order of your rules.")

### Step 4: Vet all five stars

Now we run the whole machine on every star, and record *which* tests failed.

In [ ]:
import pandas as pd

def failed_tests(tlc):
    # Collect the names of every test the signal failed
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        check_thresholds(tlc.metrics, "FA", verbose=True)
        check_thresholds(tlc.metrics, "FP", verbose=True)
    return [line.split(": ", 1)[1] for line in buf.getvalue().splitlines() if line[:2] in ("FA", "FP")]

rows = []
tlcs = {}
for name, s in stars.items():
    tlc = make_tlc(s)
    tlc.compute_flux_metrics(sim_star_props, verbose=False)
    tlcs[name] = tlc
    rows.append({
        "Star": name,
        "LEO-Vetter verdict": leo_verdict(tlc),
        "S/N": round(tlc.metrics["MES"], 1),
        "Radius (Re)": round(tlc.metrics["Rp"], 1),
        "Odd/even (sigma)": round(tlc.metrics["sig_dep"], 1),
        "Secondary (sigma)": round(tlc.metrics["sig_sec"], 1),
        "Failed tests": "; ".join(failed_tests(tlc)) or "none",
    })

pd.set_option("display.max_colwidth", 70)
results = pd.DataFrame(rows).set_index("Star")
results

### How did your verdicts compare?

LEO-Vetter has three bins (**PC**, **FP**, **FA**). Your four categories map onto them like this:

| Your verdict | LEO-Vetter verdict |
|--------------|-------------------|
| planet | PC |
| eclipsing binary | FP |
| variable star | FA |
| systematic | FA |

In [ ]:
to_leo = {"planet": "PC", "eclipsing binary": "FP", "variable star": "FA", "systematic": "FA"}

print(f"{'Star':<6}{'Your verdict':<20}{'LEO-Vetter':<12}{'Agree?'}")
print("-" * 46)
for name in stars:
    mine = my_verdicts.get(name)
    leo = results.loc[name, "LEO-Vetter verdict"]
    if mine is None:
        agree = "(no verdict entered)"
    else:
        agree = "✓" if to_leo.get(mine) == leo else "✗  <-- worth discussing!"
    print(f"{name:<6}{str(mine):<20}{str(leo):<12}{agree}")

### 🔎 Discuss: the star that slipped through

Look at **Star E** in the table. LEO-Vetter's flux-level tests passed it as a planet candidate. But in Exercise 4 *you* found something it never looked at: the background light jumps every time the "transit" happens.

The lesson is not that the tool is bad. This run only analyzed the **flux**. Vetting is a stack of evidence, and a person (or a pixel-level test) looking at a different data product can catch what a flux test cannot. Real TESS systematics are messier than our simulated one, and LEO-Vetter has further tests we haven't used here.

### Step 5: Vetting reports

LEO-Vetter also makes a one-page **report** for each signal, so a human can double-check the verdict. It collects the same diagnostics you built: the folded transit, odd vs even, and the secondary. Here are the reports for the two eclipsing binaries.

In [ ]:
from leo_vetter.plots import plot_summary

plot_summary(tlcs["B"], sim_star_props)
plt.show()

**Star B:** look at the "Odd" and "Even" panels in the middle. The odd transits are visibly deeper than the even ones, and the report prints the significance. That's your Exercise 2, drawn by a professional tool.

In [ ]:
plot_summary(tlcs["C"], sim_star_props)
plt.show()

**Star C:** look at the phase diagram. There is a slight dip at phase 0.5. That's your Exercise 3.

---
## When *Could* It Be a Planet?

Four of our five stars failed at least one test. **Star A passed everything.** Here is every individual test LEO-Vetter ran on it, one by one:

In [ ]:
from leo_vetter import thresholds as T

def scorecard(tlc):
    # List every test and whether it was triggered
    tests = [T.weak, T.invalid_transits, T.bad_shape, T.non_unique, T.chases, T.dmm, T.single_event,
             T.bad_fit, T.sinusoidal, T.unphysical_duration, T.asymmetric, T.chi, T.data_gapped,
             T.odd_even, T.vshaped, T.large, T.secondary]
    print(f"{'Failure mode checked':<48}{'Result'}")
    print("-" * 62)
    for test in tests:
        flag, message = test(tlc.metrics, T._default_thresholds)
        print(f"{message.split(': ', 1)[1]:<48}{'✗ TRIGGERED' if flag else '✓ clear'}")

print("STAR A (the planet candidate)")
scorecard(tlcs["A"])

**Every box is clear.** For a planet candidate, that means:

- **The signal is real and repeatable:** strong enough, enough transits, the same shape each time.
- **The shape is right:** flat-bottomed with a sharp ingress and egress, not a V and not a smooth wave.
- **The size is right:** a few Earth radii up to about Jupiter's size, not a star.
- **Odd and even transits agree:** the same object blocks the same star by the same amount each orbit.
- **No significant secondary eclipse**, or only a faint one consistent with a planet reflecting or emitting a little light. (Very hot, large planets can show one; the tool allows it only if it's small enough for a planet.)
- **The orbit is physically possible:** the transit duration, the period and the star's density have to fit together.

### Passing is not proof

**PC means "nothing ruled it out", not "confirmed planet".** A passing signal can still be an impostor the tests can't see: for example, a faint eclipsing binary in the same pixels (the pixel-level tests are meant to catch these), or a brown dwarf or low-mass star that happens to be Jupiter-sized. To go from candidate to confirmed planet, astronomers need follow-up:

- **Mass:** measure the star's wobble from the ground (radial velocities). A planet tugs on the star weakly; a star tugs hard.
- **High-resolution imaging:** check that there is no hidden neighbor.
- **Statistical validation:** calculate how likely each impostor scenario is, and show the planet explanation wins by a large margin.

### What about pixel-level vetting?

Everything so far used the **light curve** (a single brightness number per time stamp). TESS pixels are big (about 21 arcseconds), so a faint eclipsing binary next to the target can land in the same pixels. LEO-Vetter's **pixel-level tests** (TESS only) build *difference images* (in-transit minus out-of-transit) and check whether the dimming is centered on the target or on a neighbor. We can't run them on simulated data, because they need real TESS pixel files and an extra package (`transit-diffImage`), but the LEO-Vetter paper shows examples for known false positives.

---
# Part 3 (Take-Home): Is TIC 287196418 a Planet?

You have everything you need. In Part 1 you downloaded a real TESS light curve and found a repeating dip. Now vet it like you vetted Stars A to E.

### What's different about real data?

- **It's messier.** Don't expect numbers as clean as the simulation. A borderline result is real science.
- **LEO-Vetter needs the real star's properties** (radius, mass, temperature, density). We look these up in the TESS Input Catalog.
- **Most of the time, the answer isn't "obvious".** Write down your reasoning, not just your verdict.

> ⚠️ Part 3 reuses the functions you wrote in Part 2 (`planet_radius`, `odd_even_test`, `secondary_test`, `background_test`, `leo_verdict`). Make sure all five are completed and their cells have been run. If you got stuck, run the solution code from the hint blocks.

### Step 1: Process the Part 1 light curve

Run the same `process_lightcurve()` function on the light curve you downloaded in Part 1. (We use the 1-5 day period search, as in Part 1.)

In [ ]:
real_star = process_lightcurve(lc_part1, name=f"TIC {TIC}", period_grid=PERIOD_GRID_REAL)

print(f"Candidate period:   {real_star['per']:.4f} days")
print(f"Mid-transit:        {real_star['epo']:.3f}")
print(f"Duration:           {real_star['dur']:.2f} days")

plot_fold(real_star)
plt.show()

### Step 2: Look up the star's properties

LEO-Vetter converts depth into a planet radius and checks the orbit using the star's size and density. The next cell queries the TESS Input Catalog for TIC 287196418. (If the catalog has no value for something, it falls back to a Sun-like default and tells you.)

In [ ]:
def props_from_tic_row(row):
    # Turn one row of the TESS Input Catalog into the dictionary LEO-Vetter wants
    def val(key):
        try:
            v = row[key]
            if np.ma.is_masked(v):
                return np.nan
            v = float(v)
            return v if np.isfinite(v) else np.nan
        except (KeyError, TypeError, ValueError):
            return np.nan

    defaults = {"rad": 1.0, "mass": 1.0, "Teff": 5700.0, "logg": 4.4}
    used_default = []
    p = {}
    for key, default in defaults.items():
        v = val(key)
        if np.isnan(v):
            v = default
            used_default.append(key)
        p[key] = v
    for key, base in [("e_rad", "rad"), ("e_mass", "mass"), ("e_Teff", "Teff")]:
        v = val(key)
        p[key] = v if np.isfinite(v) else 0.1 * p[base]

    # The catalog gives density in solar units; LEO-Vetter wants g/cm^3 (the Sun is 1.41 g/cm^3)
    rho_solar = val("rho")
    if np.isnan(rho_solar):
        rho_solar = p["mass"] / p["rad"] ** 3
        used_default.append("rho")
    p["rho"] = rho_solar * 1.41

    try:
        u1, u2 = quadratic_ldc(p["Teff"], p["logg"], 0.0)
        p["u1"], p["u2"] = float(u1), float(u2)
    except Exception:
        p["u1"], p["u2"] = 0.4, 0.25
        used_default.append("limb darkening")
    return p, used_default

def get_star_props(tic):
    from astroquery.mast import Catalogs
    table = Catalogs.query_criteria(catalog="TIC", ID=tic)
    return props_from_tic_row(table[0])

real_props, defaults_used = get_star_props(TIC)
print(f"Radius: {real_props['rad']:.2f} Rsun   Mass: {real_props['mass']:.2f} Msun   "
      f"Teff: {real_props['Teff']:.0f} K   Density: {real_props['rho']:.2f} g/cm^3")
if defaults_used:
    print("Used Sun-like defaults for:", ", ".join(defaults_used))

### Step 3: Run your four tests

This helper runs the four tests you wrote in Part 2a on any star (it uses the real star's radius for the size test).

In [ ]:
def manual_vet(star, r_star_rsun=1.0):
    depth, rp = planet_radius(star, r_star_rsun)
    d_odd, d_even, s_oe = odd_even_test(star)
    d_sec, s_sec = secondary_test(star)
    excess, s_bkg = background_test(star)
    print(f"{star['name']}   (candidate period {star['per']:.3f} d)")
    print(f"  Size:       {rp:.1f} Earth radii")
    print(f"  Odd/even:   {d_odd*1e3:.2f} vs {d_even*1e3:.2f} ppt  ({s_oe:.1f} sigma)")
    print(f"  Secondary:  {d_sec*1e3:.2f} ppt  ({s_sec:.1f} sigma)")
    print(f"  Background: {s_bkg:.1f} sigma")

manual_vet(real_star, r_star_rsun=real_props["rad"])

**Before you run LEO-Vetter, write your prediction** in the cell below: planet candidate, or impostor? If an impostor, which kind, and which test gave it away?

*(Double-click this cell and write your answer here.)*

### Step 4: Run LEO-Vetter

Use the functions from Part 2b. Fill in the three blanks.

In [ ]:
# TODO 1: package the real star with make_tlc()
tlc_real = None

# TODO 2: compute the metrics: tlc_real.compute_flux_metrics(real_props, verbose=False)
#         (remember to use real_props, not sim_star_props)

# TODO 3: get the verdict with leo_verdict()
verdict_real = None

<details>
<summary>💡 Click here for the solution</summary>

```python

# Package the real star with make_tlc()
tlc_real = make_tlc(real_star)

# Compute the metrics, using the REAL star's properties
tlc_real.compute_flux_metrics(real_props, verbose=False)

# Get the verdict
verdict_real = leo_verdict(tlc_real)

```
</details>

In [ ]:
if tlc_real is None or verdict_real is None:
    print("❌ Please fill in the blanks above!")
else:
    print(f"LEO-Vetter verdict for TIC {TIC}: {verdict_real}\n")
    print("Failed tests:", "; ".join(failed_tests(tlc_real)) or "none")
    print()
    scorecard(tlc_real)

### Step 5: The report

Look at the full one-page report and compare it with your four manual tests. Do the odd/even and secondary panels agree with the numbers you computed by hand?

In [ ]:
plot_summary(tlc_real, real_props)
plt.show()

### Step 6: Write your conclusion

In the cell below, write 3-5 sentences: **What is TIC 287196418? Which tests supported your conclusion? What would you do next to confirm it?**

*(Double-click this cell and write your answer here.)*

<details>
<summary>🔑 Instructor answer key </summary>

**Instructors:** run Part 3 end to end on your own copy first, then paste the results here.

- BLS period / duration: 3.695 d, 0.15 d (t0 ≈ 1684.208)
- Manual tests: size 15.0 Earth radii (host star 1.63 Rsun); odd/even 5.25 vs 8.52 ppt (23.5σ); secondary 0.33 ppt (4.1σ); background -2.2σ
- LEO-Vetter: FP. Failed test: odd-even transit differences. Everything else clear.
- What differs from the simulation: same signature as Star B. The BLS period is most likely half the true orbital period (about 7.39 d), so the odd and even dips are different eclipses of an eclipsing binary. The size and secondary tests do not flag it, so odd/even is the test that catches it.

</details>

---
## Optional Extensions

### A. Try a different real star

Change `TIC` and `SECTOR` in Part 1 and re-run from there. To find real stars to try, go to **ExoFOP** (https://exofop.ipac.caltech.edu/tess/), which lists every TESS candidate with its current **disposition** (confirmed planet, false positive, and so on). Pick one, run the whole workflow, and check your verdict against the catalog's. When they disagree, ask why.

You may need to adjust `PERIOD_GRID_REAL`: the search range of 1-5 days suits this star, but your star's period may be shorter or longer. Make sure the range covers its published period.

### B. The mystery stars (simulated)

Two new simulated candidates. Decide what each is, then check with LEO-Vetter.

In [ ]:
mystery = {name: prepare_star(name) for name in ["M1", "M2"]}

for s in mystery.values():
    manual_vet(s)
    print()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, s in zip(axes, mystery.values()):
    plot_fold(s, ax)
plt.tight_layout()
plt.show()

In [ ]:
# What does LEO-Vetter say about the mystery stars?
for name, s in mystery.items():
    tlc = make_tlc(s)
    tlc.compute_flux_metrics(sim_star_props, verbose=False)
    print(f"Mystery star {name}:  LEO-Vetter verdict = {leo_verdict(tlc)}")
    print(f"   Failed tests: {'; '.join(failed_tests(tlc)) or 'none'}\n")

<details>
<summary>🔑 Answer key for the mystery stars</summary>

- **M1** is an **eclipsing binary**. Its period is half the true orbital period, so the odd and even eclipses have different depths (the odd/even test fails).
- **M2** is a **planet**: the same depth in odd and even transits, no secondary eclipse, no background link, and a planet-sized radius.
</details>

### C. Thresholds are choices

The cut-offs are not laws of nature. They were tuned for typical TESS stars, and you can change them. Let's decide that "planets" must be smaller than 5 Earth radii (Neptune-sized and below).

In [ ]:
# (The leading underscore means "internal", but it's handy to peek at the defaults)
from leo_vetter.thresholds import _default_thresholds

strict = dict(_default_thresholds)
strict["size"] = 5   # default is 22 Earth radii

print(f"Star A radius: {tlcs['A'].metrics['Rp']:.1f} Earth radii\n")
print("With the default threshold (22 Earth radii):")
check_thresholds(tlcs["A"].metrics, "FP", verbose=True)
print("\nWith our stricter threshold (5 Earth radii):")
check_thresholds(tlcs["A"].metrics, "FP", verbose=True, thresholds=strict)

Same light curve, different verdict, because the **question** changed. That is why automated, documented thresholds matter to scientists who measure how common planets are.

---
# Summary: What We Accomplished

## The Problem
A search algorithm found repeating dips in starlight. We needed to decide which are **planets** and which are **impostors**.

## What You Did

1. **Found a transit signal in real TESS data**: download → bin → flatten → BLS period search → phase fold
2. **Vetted five simulated stars by hand** with four tests, each aimed at a different impostor
3. **Used LEO-Vetter** to run dozens of tests automatically, read its reports, and see why a passing signal is a *candidate*, not a confirmed planet
4. **(Take-home)** Vetted the real star from Part 1

## Key Concepts

| Concept | What It Means | Why It Matters |
|---------|---------------|----------------|
| **Transit depth** | Fraction of starlight blocked | Gives the size of the object |
| **Odd/even test** | Compare alternating transits | Exposes eclipsing binaries and wrong periods |
| **Secondary eclipse** | A dip at orbit fraction 0.5 | Planets rarely show a deep one; binaries do |
| **Background check** | Does the dip follow the scattered light? | Exposes instrumental glitches |
| **Pass/fail thresholds** | Cut-offs applied to metrics | Make vetting repeatable, but they are choices |
| **Candidate vs. confirmed** | Passing vetting is not proof | Confirmation needs follow-up (for example, measuring the star's wobble from the ground) |

## Go Further

- LEO-Vetter on PyPI: https://pypi.org/project/leo-vetter/ (the project page links to full tutorials)
- Paper: Kunimoto et al. 2025, *LEO-Vetter: Fully Automated Flux- and Pixel-Level Vetting of TESS Planet Candidates to Support Occurrence Rates*, The Astronomical Journal 170, 280 (arXiv:2509.10619)
- Check real candidates at ExoFOP: https://exofop.ipac.caltech.edu/tess/

---

# 🎉 Congratulations!

You've vetted transit signals like an exoplanet astronomer.